#### Analysis of EURO-CORDEX annual mean data (in progress)

- compare EURO-CORDEX model data over selected time periods to investigate INDICATOR sensitivity to model selection
- includes listing the available data (currently dummie names model1, model2, ...) and scenarios, lists available time steps
- time period and scenario for comparison can be selected by user
- Plot includes all CSAs in a panel with a joint color scale (relevant for difference plots)
- Output includes panel plots and nc-files

To Do:
- flexible difference plots
- Boxplots
- table output

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

# --- find repo root (walk up until .git or outputs/ exists) ---
NOTEBOOK_DIR = Path.cwd()

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "outputs").exists():
            return p
    return start

REPO_ROOT = find_repo_root(NOTEBOOK_DIR)

# --- make notebooks/ importable so `notebooks/_lib` can be imported as `_lib` ---
NOTEBOOKS_ROOT = REPO_ROOT / "notebooks"
if str(NOTEBOOKS_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_ROOT))

from _lib.fua_boundaries import load_fua_layer, make_load_fua_boundary, CSA_LIST

# ---- EURO-CORDEX testing paths ----
EUROCORDEX_TESTDATA_ROOT = REPO_ROOT / "2601_EURO_CORDEX_testing_data"
EUROCORDEX_TEST_UNZIP_ROOT = EUROCORDEX_TESTDATA_ROOT / "unzippedIT"
EUROCORDEX_TEST_OUTPUT_ROOT = REPO_ROOT / "outputs" / "eurocordex_testing"

DERIVED_DIR = EUROCORDEX_TEST_OUTPUT_ROOT / "derived"
TABLES_DIR  = EUROCORDEX_TEST_OUTPUT_ROOT / "tables"
PLOTS_DIR   = EUROCORDEX_TEST_OUTPUT_ROOT / "plots"

# FUA boundaries
FUA_DIR = REPO_ROOT / "shapefile" / "UI-boundaries-FUA"

# Create directories (safe: ignored by git)
for d in [EUROCORDEX_TEST_UNZIP_ROOT, EUROCORDEX_TEST_OUTPUT_ROOT, DERIVED_DIR, TABLES_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# CSA ↔ FUA name mapping
FUA_MAPPING = {
    "Prague": "Praha",
    "Funen-Odense": "Odense",
    "Athens": "Athina",
    "Birmingham": "West Midlands urban area",
}

_fua_all = load_fua_layer(FUA_DIR)
load_fua_boundary = make_load_fua_boundary(_fua_all, name_field="FUA_NAME", mapping=FUA_MAPPING)

print("REPO_ROOT:", REPO_ROOT)
print("NOTEBOOKS_ROOT:", NOTEBOOKS_ROOT)
print("EUROCORDEX_TESTDATA_ROOT:", EUROCORDEX_TESTDATA_ROOT)
print("EUROCORDEX_TEST_UNZIP_ROOT:", EUROCORDEX_TEST_UNZIP_ROOT)
print("EUROCORDEX_TEST_OUTPUT_ROOT:", EUROCORDEX_TEST_OUTPUT_ROOT)
print("Loaded FUA layer rows:", len(_fua_all))
print("FUA columns:", list(_fua_all.columns))


REPO_ROOT: C:\Users\reinhvlr\OneDrive\CARMINE-T2.4
NOTEBOOKS_ROOT: C:\Users\reinhvlr\OneDrive\CARMINE-T2.4\notebooks
EUROCORDEX_TESTDATA_ROOT: C:\Users\reinhvlr\OneDrive\CARMINE-T2.4\2601_EURO_CORDEX_testing_data
EUROCORDEX_TEST_UNZIP_ROOT: C:\Users\reinhvlr\OneDrive\CARMINE-T2.4\2601_EURO_CORDEX_testing_data\unzippedIT
EUROCORDEX_TEST_OUTPUT_ROOT: C:\Users\reinhvlr\OneDrive\CARMINE-T2.4\outputs\eurocordex_testing
Loaded FUA layer rows: 672
FUA columns: ['OBJECTID', 'FUA_CODE', 'FUA_NAME', 'COUNTRY_CO', 'Shape_Leng', 'Shape_Area', 'geometry']


In [ ]:
import re
import tarfile
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd

In [ ]:
archives = sorted([
    p for p in EUROCORDEX_TESTDATA_ROOT.iterdir()
    if p.is_file() and p.name.lower().endswith((".tar.gz", ".tgz"))
])

print("Found archives:", [a.name for a in archives])

for tar_fp in archives:
    stem = tar_fp.name
    stem = stem[:-7] if stem.lower().endswith(".tar.gz") else stem[:-4]  # strip .tar.gz / .tgz
    target = EUROCORDEX_TEST_UNZIP_ROOT / stem
    target.mkdir(parents=True, exist_ok=True)

    if any(target.rglob("*")):
        print("Already extracted:", tar_fp.name, "→", target.name)
        continue

    print("Extracting:", tar_fp.name, "→", target)
    with tarfile.open(tar_fp, mode="r:*") as tf:
        tf.extractall(path=target)

In [ ]:
# Matches stems like: pr_cordex_model02_rcp85
# (works even if more tokens follow after scenario)
FNAME_RE = re.compile(
    r"^(?P<var>[a-z0-9]+)_(?P<dataset>cordex)_(?P<model>model\d+?)_(?P<scenario>[a-z0-9]+)",
    re.IGNORECASE,
)

def parse_cordex_stem(fp: Path) -> dict:
    """
    Parse EURO-CORDEX test filenames like:
      pr_cordex_model02_rcp85(.nc)

    Returns:
      var, dataset, model_id, scenario
    If parsing fails, returns None values.
    """
    stem = fp.stem  # drops .nc
    m = FNAME_RE.match(stem)
    if not m:
        return {"var": None, "dataset": None, "model_id": None, "scenario": None}

    d = m.groupdict()
    return {
        "var": d["var"].lower(),
        "dataset": d["dataset"].lower(),
        "model_id": d["model"].lower(),     # keep model02 as-is
        "scenario": d["scenario"].lower(),  # e.g., rcp85
    }

# quick self-test (edit if you want)
_test = Path("pr_cordex_model02_rcp85.nc")
print("Test parse:", _test.name, "→", parse_cordex_stem(_test))


In [ ]:
# =============================================================================
# 4 — Discover NetCDFs and summarize availability
# =============================================================================

# 1) discover files
nc_files = sorted([
    p for p in EUROCORDEX_TEST_UNZIP_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in (".nc", ".nc4", ".cdf")
])

print("UNZIP_ROOT:", EUROCORDEX_TEST_UNZIP_ROOT)
print("Found NetCDFs:", len(nc_files))

if not nc_files:
    warnings.warn("No NetCDF files found. Check extraction and UNZIP_ROOT.")
else:
    print("Example filenames:")
    for p in nc_files[:10]:
        print(" -", p.name)

# 2) parse filename tokens
rows = []
for fp in nc_files:
    meta = parse_cordex_stem(fp)
    rows.append({
        "file_name": fp.name,
        "path": str(fp),
        **meta,
    })

files_df = pd.DataFrame(rows)

# 3) availability summary: what exists?
avail = (
    files_df
    .groupby(["var", "scenario", "model_id"], dropna=False)
    .size()
    .reset_index(name="n_files")
    .sort_values(["var", "scenario", "model_id"])
)

display(avail)

# 4) lists for user selection later
AVAILABLE_VARS = sorted([v for v in files_df["var"].dropna().unique()])
AVAILABLE_SCENARIOS = sorted([s for s in files_df["scenario"].dropna().unique()])
AVAILABLE_MODELS = sorted([m for m in files_df["model_id"].dropna().unique()])

print("AVAILABLE_VARS:", AVAILABLE_VARS)
print("AVAILABLE_SCENARIOS:", AVAILABLE_SCENARIOS)
print("AVAILABLE_MODELS:", AVAILABLE_MODELS)

# 5) quick parse QA (helps catch weird filenames)
n_bad = int(files_df["model_id"].isna().sum())
if n_bad:
    warnings.warn(f"{n_bad} file(s) did not match the expected naming pattern.")
    display(files_df.loc[files_df["model_id"].isna(), ["file_name", "path"]].head(20))


In [ ]:
# =============================================================================
# 5 — USER INPUT (selection for plotting / comparisons)
# =============================================================================

print("AVAILABLE_VARS:", AVAILABLE_VARS)
print("AVAILABLE_SCENARIOS:", AVAILABLE_SCENARIOS)
print("AVAILABLE_MODELS:", AVAILABLE_MODELS)

# --- Core selection ---
VAR_NAME  = AVAILABLE_VARS[0] if AVAILABLE_VARS else "pr"
SCENARIO  = "rcp26" if "rcp26" in AVAILABLE_SCENARIOS else (AVAILABLE_SCENARIOS[0] if AVAILABLE_SCENARIOS else None)

# Choose models to include (subset of AVAILABLE_MODELS)
SELECT_MODELS = AVAILABLE_MODELS[:]  # default = all available

# Pairwise comparisons (B - A); adjust freely
COMPARE_PAIRS = []
if len(SELECT_MODELS) >= 2:
    COMPARE_PAIRS = list(zip(SELECT_MODELS[:-1], SELECT_MODELS[1:]))

# --- What derived product to work with ---
# Must match your derived-product naming later (we'll wire this up)
PRODUCT = "period_mean"   # options you will support: "annual_mean", "period_mean"

# Only relevant if PRODUCT == "period_mean"
PERIOD_LABEL = "2036-2065"   # set to your actual period labels later

# --- Plot options ---
DO_PANEL_PLOTS = True
DO_DIFFERENCE_MAPS = True
DO_BOX_PLOTS = True

print("\n--- USER INPUT SUMMARY ---")
print("VAR_NAME:", VAR_NAME)
print("SCENARIO:", SCENARIO)
print("SELECT_MODELS:", SELECT_MODELS)
print("COMPARE_PAIRS (B-A):", COMPARE_PAIRS)
print("PRODUCT:", PRODUCT)
print("PERIOD_LABEL:", PERIOD_LABEL)


In [ ]:
# =============================================================================
# 6 — Select input files based on USER INPUT
# =============================================================================

if SCENARIO is None:
    raise ValueError("SCENARIO is None. Check AVAILABLE_SCENARIOS and user input.")

sel_files_df = files_df[
    (files_df["var"] == VAR_NAME) &
    (files_df["scenario"] == SCENARIO) &
    (files_df["model_id"].isin(SELECT_MODELS))
].copy()

sel_files_df = sel_files_df.sort_values(["model_id", "file_name"]).reset_index(drop=True)

print("Selected files:", len(sel_files_df))
display(sel_files_df[["model_id", "scenario", "var", "file_name", "path"]].head(20))

# sanity: do we have at least one file per selected model?
counts = sel_files_df["model_id"].value_counts().reindex(SELECT_MODELS, fill_value=0)
print("\nFiles per selected model:")
print(counts)

missing_models = [m for m, n in counts.items() if n == 0]
if missing_models:
    warnings.warn(f"No files found for model(s): {missing_models}. Check naming / selection.")


In [ ]:
# =============================================================================
# 7 — Time coverage overview per model (lightweight read)
# =============================================================================

def _time_summary_one(fp: Path) -> dict:
    row = {
        "model_id": parse_cordex_stem(fp)["model_id"],
        "file_name": fp.name,
        "path": str(fp),
        "n_time": None,
        "time_start": None,
        "time_end": None,
    }
    try:
        # use decode_times=True (works for standard and cftime calendars)
        ds = xr.open_dataset(fp, decode_times=True)
        try:
            if "time" in ds.sizes:
                row["n_time"] = int(ds.sizes["time"])
            if "time" in ds.coords and ds.sizes.get("time", 0) > 0:
                row["time_start"] = str(ds["time"].values[0])
                row["time_end"]   = str(ds["time"].values[-1])
        finally:
            ds.close()
    except Exception as e:
        row["time_start"] = f"ERROR: {e}"
    return row

# Build file-level time summary for the selected files
time_rows = []
for p in sel_files_df["path"]:
    time_rows.append(_time_summary_one(Path(p)))

time_df = pd.DataFrame(time_rows)

print("Files summarized:", len(time_df))
display(time_df[["model_id", "n_time", "time_start", "time_end", "file_name"]].head(30))

# Model-level summary
model_time = (
    time_df
    .groupby("model_id", dropna=False)
    .agg(
        n_files=("file_name", "count"),
        min_n_time=("n_time", "min"),
        max_n_time=("n_time", "max"),
        time_start=("time_start", "min"),
        time_end=("time_end", "max"),
    )
    .reset_index()
    .sort_values("model_id")
)

print("\n--- Time coverage per model ---")
display(model_time)

# Optional: store for later use (e.g., in UI / selection validation)
MODEL_TIME_COVERAGE = model_time


In [ ]:
# =============================================================================
# 8a — Period definitions (annual values already in input!)
# =============================================================================

# Define the periods you want to average over
PERIODS = {
    "2036-2065": ("2036-01-01", "2065-12-31"),
    # add more later if needed
    # "2021-2050": ("2021-01-01", "2050-12-31"),
}

def subset_period(ds: xr.Dataset, start: str, end: str) -> xr.Dataset:
    """
    Subset by time slice.
    Works for standard datetime + cftime because we slice with strings.
    """
    if "time" not in ds.coords:
        raise KeyError("Dataset has no 'time' coordinate.")
    return ds.sel(time=slice(start, end))

In [ ]:
# =============================================================================
# 8b — Derive PERIOD MEANS from annual data (one file)
# =============================================================================

def derive_period_means_for_file(fp: Path) -> list[dict]:
    """
    Input files already contain ANNUAL values along 'time'.
    We compute PERIOD MEANS only (mean over selected years).
    Writes one derived NetCDF per (model, scenario, var, period).
    """
    meta = parse_cordex_stem(fp)
    var = meta["var"]
    if var is None:
        raise ValueError(f"Could not parse variable name from filename: {fp.name}")

    rows = []
    ds = xr.open_dataset(fp, decode_times=True)

    try:
        if var not in ds:
            raise KeyError(f"Variable '{var}' not found in dataset vars: {list(ds.data_vars)}")

        da = ds[var]

        # Compute mean over time for each requested period
        for label, (start, end) in PERIODS.items():
            ds_p = subset_period(ds, start, end)
            if ds_p.sizes.get("time", 0) == 0:
                warnings.warn(f"No timesteps in period {label} for file {fp.name}")
                continue

            da_p = ds_p[var].mean("time")

            out_p = (
                DERIVED_DIR /
                f"{var}_cordex_{meta['model_id']}_{meta['scenario']}_periodmean_{label}.nc"
            )
            da_p.to_netcdf(out_p)

            rows.append({
                "var": var,
                "scenario": meta["scenario"],
                "model_id": meta["model_id"],
                "product": "period_mean",
                "period": label,
                "source_path": str(fp),
                "out_path": str(out_p),
                "n_time_in_period": int(ds_p.sizes.get("time", 0)),
            })

    finally:
        ds.close()

    return rows

In [ ]:
# =============================================================================
# 8c — Bulk period-mean derivation + results_df catalog
# =============================================================================

all_results = []

for p in sel_files_df["path"]:
    fp = Path(p)
    print("Period means from:", fp.name)
    all_results.extend(derive_period_means_for_file(fp))

results_df = pd.DataFrame(all_results)

print("Derived products:", len(results_df))
display(results_df.sort_values(["var", "scenario", "model_id", "period"]).reset_index(drop=True))

# Quick sanity check
if results_df.empty:
    warnings.warn("results_df is empty — check PERIODS and whether time slicing matched your data.")
else:
    print("\nCounts by model/period:")
    display(results_df.groupby(["model_id", "period"]).size().reset_index(name="n"))

In [ ]:
# =============================================================================
# PANEL PLOT — 7 CSAs (FUA zoom), one period_mean + one model (dummy naming)
# =============================================================================
import matplotlib.pyplot as plt
import warnings

# ---- USER CHOICE FOR THE PANEL ----
PANEL_PRODUCT = "period_mean"   # we now store product + period separately
PANEL_PERIOD  = PERIOD_LABEL    # e.g. "1981-2010" (from your USER INPUT cell)
PANEL_MODEL_FILTER = None       # substring to pick one model, e.g. "model02"
CMAP = "viridis"
ROBUST = True
PCTL = (2, 98)
ADD_BOUNDARY = True
DPI = 200

assert "results_df" in globals() and not results_df.empty, "results_df missing (run period mean calculations)."
assert "CSA_LIST" in globals() and len(CSA_LIST) > 0, "CSA_LIST missing."
assert "load_fua_boundary" in globals(), "load_fua_boundary missing (FUA helper not loaded)."

# --- helper: get lon/lat coord names and arrays ---
def get_lon_lat(da: xr.DataArray):
    for lon, lat in [("lon","lat"), ("longitude","latitude"), ("LON","LAT"), ("nav_lon","nav_lat")]:
        if lon in da.coords and lat in da.coords:
            return da[lon].values, da[lat].values
    return None, None

def dissolve_to_lonlat_polygon(csa: str):
    gdf = load_fua_boundary(csa)
    if gdf is None or gdf.empty:
        return None
    bg = gdf
    if getattr(bg, "crs", None) is not None:
        bg = bg.to_crs("EPSG:4326")
    return bg.geometry.union_all()

# --- pick one derived file matching product + period (+ optional model filter) ---
sub = results_df[
    (results_df["product"] == PANEL_PRODUCT) &
    (results_df["period"] == PANEL_PERIOD)
].copy()

if sub.empty:
    raise ValueError(
        f"No derived outputs for product='{PANEL_PRODUCT}' and period='{PANEL_PERIOD}'. "
        f"Available products: {sorted(results_df['product'].unique())}; "
        f"Available periods: {sorted([p for p in results_df['period'].dropna().unique()])}"
    )

if PANEL_MODEL_FILTER is not None:
    sub = sub[sub["model_id"].astype(str).str.contains(PANEL_MODEL_FILTER, case=False, na=False)].copy()
    if sub.empty:
        raise ValueError(f"No derived outputs match PANEL_MODEL_FILTER='{PANEL_MODEL_FILTER}'.")

# choose the first match
picked = sub.iloc[0]
DERIVED_PATH = Path(picked["out_path"])
MODEL_ID = picked.get("model_id")

print("Using derived file:", DERIVED_PATH.name)
print("Model:", MODEL_ID)
print("Product/period:", PANEL_PRODUCT, PANEL_PERIOD)

# --- load map (period_mean has no time/year dim, but keep safeguards) ---
ds = xr.open_dataset(DERIVED_PATH, decode_times=True)
try:
    da = ds[VAR_NAME].squeeze()

    # If some derived product still has year/time dims, pick first slice
    year_val = None
    if "year" in da.dims:
        year_val = int(da["year"].values[0])
        da = da.isel(year=0)
    if "time" in da.dims:
        da = da.isel(time=0)

    lon, lat = get_lon_lat(da)
    if lon is None or lat is None:
        raise RuntimeError("No lon/lat coordinates found in derived product. Needed for map plotting/zoom.")

    # --- compute shared color scale across all CSAs (based on each CSA-masked pixels) ---
    all_vals = []
    csa_polys = {}

    for csa in CSA_LIST:
        poly = dissolve_to_lonlat_polygon(csa)
        if poly is None:
            warnings.warn(f"Skip CSA={csa} (no polygon)")
            continue
        csa_polys[csa] = poly

        # mask values within polygon using shapely (vectorized preferred)
        try:
            from shapely import contains_xy
            if lon.ndim == 1 and lat.ndim == 1:
                xx, yy = np.meshgrid(lon, lat)
                m = contains_xy(poly, xx, yy)
            else:
                m = contains_xy(poly, lon, lat)
        except Exception:
            # fallback to shapely.vectorized (older shapely)
            from shapely import vectorized
            if lon.ndim == 1 and lat.ndim == 1:
                xx, yy = np.meshgrid(lon, lat)
                m = vectorized.contains(poly, xx, yy)
            else:
                m = vectorized.contains(poly, lon, lat)

        arr = np.asarray(da.values, dtype="float64")
        if m.shape == arr.shape and np.any(m):
            all_vals.append(arr[m])

    if not all_vals:
        raise RuntimeError("No CSA masks produced any values. Check polygons overlap lon/lat grid.")

    all_vals = np.concatenate(all_vals)
    all_vals = all_vals[np.isfinite(all_vals)]

    if ROBUST and all_vals.size > 0:
        vmin, vmax = np.percentile(all_vals, PCTL)
    else:
        vmin, vmax = float(np.nanmin(all_vals)), float(np.nanmax(all_vals))

    print("Color scale:", vmin, "to", vmax)

    # --- plot panel (7 CSAs) ---
    n = len(CSA_LIST)
    ncols = 4
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4*ncols, 3.5*nrows))
    axes = np.array(axes).reshape(-1)

    im_last = None

    for i, csa in enumerate(CSA_LIST):
        ax = axes[i]
        poly = csa_polys.get(csa, None)
        if poly is None:
            ax.set_axis_off()
            ax.set_title(f"{csa} (no boundary)")
            continue

        arr = np.asarray(da.values, dtype="float64")

        # draw full map, then zoom to CSA bounds
        if lon.ndim == 1 and lat.ndim == 1:
            extent = [float(np.nanmin(lon)), float(np.nanmax(lon)),
                      float(np.nanmin(lat)), float(np.nanmax(lat))]
            im = ax.imshow(arr, origin="lower", extent=extent, cmap=CMAP, vmin=vmin, vmax=vmax)
        else:
            im = ax.pcolormesh(lon, lat, arr, shading="auto", cmap=CMAP, vmin=vmin, vmax=vmax)

        im_last = im

        # overlay boundary + zoom
        if ADD_BOUNDARY:
            try:
                x, y = poly.exterior.xy
                ax.plot(x, y, linewidth=1.2, color="black")
            except Exception:
                try:
                    for geom in getattr(poly, "geoms", []):
                        x, y = geom.exterior.xy
                        ax.plot(x, y, linewidth=1.2, color="black")
                except Exception as e:
                    warnings.warn(f"Boundary plot failed for {csa}: {e}")

        minx, miny, maxx, maxy = poly.bounds
        pad_x = (maxx - minx) * 0.15
        pad_y = (maxy - miny) * 0.15
        ax.set_xlim(minx - pad_x, maxx + pad_x)
        ax.set_ylim(miny - pad_y, maxy + pad_y)

        ax.set_title(csa)
        ax.set_xlabel("lon")
        ax.set_ylabel("lat")

    for j in range(len(CSA_LIST), len(axes)):
        axes[j].set_axis_off()

    # shared colorbar in dedicated axis
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
    cbar = fig.colorbar(im_last, cax=cax)

    units = str(da.attrs.get("units", "")).strip()
    if units:
        cbar.set_label(units)

    sup = f"{VAR_NAME} — {PANEL_PRODUCT} ({PANEL_PERIOD})"
    if year_val is not None:
        sup += f" (year={year_val})"
    sup += f"\n{MODEL_ID}"
    fig.suptitle(sup, y=0.995)

    # output file
    out_dir = EUROCORDEX_TEST_OUTPUT_ROOT / "plots" / "panel"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_png = out_dir / f"{VAR_NAME}_{PANEL_PRODUCT}_{PANEL_PERIOD}_panel_{MODEL_ID}.png"

    fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
    fig.subplots_adjust(right=0.9)
    print("Saved panel:", out_png)

    plt.show()

finally:
    ds.close()
